# Unitary Manifold — Notebook 02: Holographic Boundary Dynamics

This notebook demonstrates **Pillar 4 — Holography**:
- Project bulk fields onto the holographic boundary screen
- Compute Bekenstein–Hawking entropy `S = A / 4G`
- Evolve boundary metric `h_ab` via `∂_t h_ab = −2K_ab + θ_ab + ω_ab`
- Verify information conservation via Gauss's theorem

**Repository:** https://github.com/wuzbak/Unitary-Manifold-

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))

import numpy as np
import matplotlib.pyplot as plt

from src.core.evolution import FieldState, step
from src.holography.boundary import (
    BoundaryState, entropy_area, boundary_area,
    evolve_boundary, information_conservation_check
)
from src.core.evolution import information_current

plt.rcParams['figure.dpi'] = 100

## 1 · Initialise bulk + boundary

In [ ]:
N, dx = 64, 0.1
bulk = FieldState.flat(N=N, dx=dx, rng=np.random.default_rng(7))
bdry = BoundaryState.from_bulk(bulk.g, bulk.B, bulk.phi, bulk.dx)

S0 = entropy_area(bdry.h)
A0 = boundary_area(bdry.h)
print(f"Initial boundary area     A = {A0:.4f}")
print(f"Initial holographic entropy S = A/4G = {S0:.4f}")
print(f"Induced metric h shape: {bdry.h.shape}  (M=N boundary points, 2×2 metric)")

## 2 · Co-evolve bulk and boundary for 200 steps

In [ ]:
dt, n_steps = 1e-3, 200

entropy_history = [S0]
area_history    = [A0]
conserv_history = []

for _ in range(n_steps):
    bulk = step(bulk, dt)
    bdry = evolve_boundary(bdry, bulk, dt)

    S = entropy_area(bdry.h)
    A = boundary_area(bdry.h)
    entropy_history.append(S)
    area_history.append(A)

    J_bulk = information_current(bulk.g, bulk.phi, bulk.dx)
    res = information_conservation_check(J_bulk, bdry.J_bdry, bulk.dx)
    conserv_history.append(res)

print(f"Final time: t = {bulk.t:.3f}")
print(f"Final entropy S = {entropy_history[-1]:.4f}  (initial: {S0:.4f})")
print(f"Entropy is monotonically non-decreasing: {all(b >= a - 1e-12 for a, b in zip(entropy_history, entropy_history[1:]))}")

## 3 · Plot boundary entropy and area evolution

In [ ]:
times = np.arange(len(entropy_history)) * dt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(times, entropy_history, color='royalblue')
ax1.set_xlabel('time')
ax1.set_ylabel('S = A / 4G')
ax1.set_title('Holographic Entropy')

ax2.plot(times, area_history, color='darkorange')
ax2.set_xlabel('time')
ax2.set_ylabel('A')
ax2.set_title('Boundary Area')

plt.suptitle('Entropy-Area Law — Bekenstein–Hawking S = A/4G')
plt.tight_layout()
plt.show()

## 4 · Information conservation check

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(np.arange(1, n_steps + 1) * dt, conserv_history, color='crimson')
ax.set_xlabel('time')
ax.set_ylabel('relative residual')
ax.set_title('Information Conservation Residual  |∫∇·J dV − ∮J·n dA| / charge')
plt.tight_layout()
plt.show()

print(f"Mean conservation residual: {np.mean(conserv_history):.4e}")
print(f"Max  conservation residual: {np.max(conserv_history):.4e}")

## 5 · Spatial profile of the induced boundary metric

In [ ]:
x = np.arange(N) * dx

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x, bdry.h[:, 0, 0], label='h₁₁')
axes[0].plot(x, bdry.h[:, 1, 1], label='h₂₂', linestyle='--')
axes[0].set_xlabel('x')
axes[0].set_ylabel('h_ab')
axes[0].set_title('Induced boundary metric components')
axes[0].legend()

axes[1].plot(x, bdry.J_bdry, color='green')
axes[1].set_xlabel('x')
axes[1].set_ylabel('J_bdry')
axes[1].set_title('Normal information flux on boundary')

plt.tight_layout()
plt.show()